# 2.2 Tools and structured outputs

An agent that makes ad keyframes will, sooner or later:

- call its image tool with a size the model doesn't support,
- wait less than the image takes,
- ask for the same image twice.

Each of those can cost GPU time for nothing. Here we design the image tool so none of them does.

You're done when:

- the model fixes an invalid call after one validation retry,
- a second identical `generate_image` call is a cache hit, with no GPU time and no cost.

One section per demo. Run the cells top to bottom.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE = "02-multi-agent-media-workflows"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "tools.py").exists())
sys.path.insert(0, str(CODE))
ASSETS = CODE.parent / "assets"

## Choose a mode

`MODE` decides which LLM the demos talk to. The image backend is a placeholder in both modes: same arguments as a real Endpoint, a flat PNG back.

| `MODE` | LLM | Image backend | Network | Cost |
| --- | --- | --- | --- | --- |
| `"fake"` | A scripted model: a fixed list of replies | Placeholder PNG | None | Free |
| `"fake-gpu"` | A real LLM on Token Factory, for the validation and plan demos | Placeholder PNG | Token Factory only | A few cents of tokens [measure] |

`"fake-gpu"` means the GPU is fake, not the LLM. The timeout and cache demos always use the scripted model, so they behave the same in both modes.

To switch, edit the cell below and run it again. No kernel restart needed.

In [ ]:
from fake import block_real_models

MODE = "fake"  # "fake" or "fake-gpu"

# In "fake" mode, any request to a real LLM fails, so a fake run can't spend tokens.
# The switch is process-wide, so this line also turns it back on for "fake-gpu".
block_real_models(MODE == "fake")
print(f"MODE = {MODE!r}: real LLM requests {'blocked' if MODE == 'fake' else 'allowed'}")

Shared setup for every demo:

- `brief`: the ad brief from `assets/brief.yaml`.
- `new_deps()`: a fresh run folder, trace and placeholder image backend. `sleep_s` is how long the "GPU" takes per image.
- `show()`: one line per trace event, then how many requests the "GPU" received, including requests the client gave up on.

The tool itself, `generate_image`, lives in the shared `code/tools.py`. Homework 2 imports it too.

In [ ]:
from pydantic_ai import Agent, Tool

from fake import PlaceholderImages, call, scripted_model
from models import Brief
from tools import Store, ToolDeps, generate_image, token_factory_model
from tracing import Tracer, new_run_id, run_traced

brief = Brief.from_yaml(ASSETS / "brief.yaml")


def new_deps(sleep_s: float = 0.2) -> ToolDeps:
    """A fresh run folder, trace and placeholder image backend."""
    tracer = Tracer(new_run_id())
    return ToolDeps(tracer, PlaceholderImages(sleep_s), Store(tracer.folder))


def show(deps: ToolDeps) -> None:
    """Print the trace of a demo, one line per event."""
    for e in deps.tracer.events():
        print(
            f"{e.kind:14} {e.name:16} cache_hit={e.cache_hit!s:5} t_gen_s={e.t_gen_s} cost={e.cost_usd}"
        )
    print("GPU requests received:", deps.images.requests)

## The tool signature is the schema

The model never sees your Python. It sees a JSON schema built from the signature and the docstring, resent every turn.

`generate_image` in `tools.py`:

```python
Size = Literal["832x480", "480x832", "1024x1024"]
Prompt = Annotated[str, Field(min_length=10, max_length=600)]


async def generate_image(
    ctx: RunContext[ToolDeps],
    prompt: Prompt,
    seed: int = SANA.seed,
    size: Size = "832x480",
) -> AssetRef:
    """Generate one keyframe image and return a reference to the PNG.

    Args:
        prompt: One scene, described visually. No on-screen text.
        seed: Change it to get a different image for the same prompt.
        size: WIDTHxHEIGHT in pixels. 832x480 matches the video frame.
    """
```

The cell below prints the schema the model receives. Look for:

- `size` as an `enum` of three values,
- `prompt` with its length bounds,
- each argument with its sentence from the docstring.

The docstring is a prompt: write it for the model.
PydanticAI checks the types before the function runs, so a bad argument never reaches the GPU.

In [ ]:
import json

from pydantic_ai.messages import ModelResponse, TextPart
from pydantic_ai.models.function import FunctionModel


def print_schema(messages, info):
    print(json.dumps(info.function_tools[0].parameters_json_schema, indent=1))
    return ModelResponse(parts=[TextPart("ok")])


await Agent(FunctionModel(print_schema), deps_type=ToolDeps, tools=[generate_image]).run("show")

## Three kinds of retry

"Retry on error" pays twice when the error happened after the GPU started. So `tools.py` keeps three kinds apart:

| Kind | Example | What we do | What it costs |
| --- | --- | --- | --- |
| Validation retry | `size="1000x1000"` | The error goes back to the model, and it fixes the call | One more LLM turn, no GPU |
| Transport retry | Connect timeout, `502`/`503` while the Endpoint loads its weights | Back off and retry, at most 3 times | Nothing: the request never reached the model |
| Uncertain submission | Read timeout, a connection dropped after sending | Never resend blindly. Record the request, check its status before any resend | Maybe a GPU run you can't see |

### Validation retry

A scripted model gets the size wrong first, then fixes it.
With `"fake-gpu"`, the real model gets "Use size 1000x1000" in the prompt instead.

Expect two model requests, one tool call and one GPU request: the bad call never ran.

In [ ]:
deps = new_deps()
model = (
    scripted_model(
        call("generate_image", prompt=brief.shot, size="1000x1000"),  # not an allowed size
        call("generate_image", prompt=brief.shot, size="832x480"),
        "Keyframe ready.",
    )
    if MODE == "fake"
    else token_factory_model()
)
agent = Agent(model, deps_type=ToolDeps, tools=[generate_image])
await run_traced(
    agent,
    f"Make the keyframe for: {brief.shot}. Use size 1000x1000.",
    tracer=deps.tracer,
    deps=deps,
)
show(deps)

With `"fake-gpu"`, count the model requests before the tool call: that's how many retries the real model needed. In our run: [measure].

## Tool timeouts

PydanticAI treats a [tool timeout](https://pydantic.dev/docs/ai/tools-toolsets/tools-advanced/#tool-timeout) as a retryable failure, counted toward the tool's retries.

For a GPU tool that's a trap:

1. The client stops waiting.
2. The GPU keeps working.
3. The retry sends a second request, and you pay for both.

The demo makes a "GPU" that takes 3 s per image, and tries a 1 s and a 10 s timeout.

In [ ]:
from pydantic_ai import UnexpectedModelBehavior

for timeout in (1, 10):
    deps = new_deps(sleep_s=3)  # this "GPU" takes 3 s per image
    agent = Agent(
        scripted_model(*[call("generate_image", prompt=brief.shot)] * 2, "Keyframe ready."),
        deps_type=ToolDeps,
        tools=[Tool(generate_image, timeout=timeout, max_retries=1)],
    )
    try:
        await run_traced(agent, "Make the keyframe.", tracer=deps.tracer, deps=deps)
    except UnexpectedModelBehavior as exc:
        print("Gave up:", exc)
    print(f"timeout={timeout} s -> GPU requests: {deps.images.requests}")

Two strategies, by how long the tool runs:

- **Short tools** (a Sana image, seconds): set the timeout above the measured p95 generation time. Sana on the course L40S: p95 [measure], so the course uses [proposed: 2 × p95].
- **Long tools** (a video, minutes): don't wait inside the tool at all. Submit, return, and check back later.

## Cache keys: reuse a completed result

The model, a retry or a rerun of your script will call the tool with the same arguments again.

The spec hash names everything that changes the output. From `spec_hash()` in `tools.py`:

```python
spec = {
    "model": model,
    "model_revision": model_revision,
    "image": image,  # container image of the Endpoint
    "prompt": prompt,
    "seed": seed,
    "params": params,
    "inputs": [file_sha256(p) for p in inputs],
    "adapter": adapter,
}
canonical = json.dumps(spec, sort_keys=True, separators=(",", ":"))
return hashlib.sha256(canonical.encode()).hexdigest()
```

The image file is named after the hash. Before any GPU call, the tool checks whether that file exists:

```python
if store.exists(name):  # same spec as a finished generation: reuse it
    return AssetRef(cache_hit=True, **ref)
```

The cell calls the tool twice with the same spec. Expect the second `tool_call` to be a cache hit: `cost_usd` 0, no `t_gen_s`, one GPU request in total.

In [ ]:
deps = new_deps()
same = call("generate_image", prompt=brief.shot, seed=7)
agent = Agent(
    scripted_model(same, call("generate_image", prompt=brief.shot, seed=7), "Done."),
    deps_type=ToolDeps,
    tools=[generate_image],
)
result = await run_traced(agent, "Make the keyframe twice.", tracer=deps.tracer, deps=deps)
show(deps)

**Exercise:** the cell below flips one byte of the keyframe and hashes both files as inputs of a video spec.
Explain why the hashes differ although the model, prompt, seed and params are the same.

That's why a tool that takes a file as input hashes the file's bytes, not its name.

In [ ]:
from tools import spec_hash

keyframe = next(deps.store.folder.glob("keyframe-*.png"))
edited = keyframe.with_name("keyframe-edited.png")
data = bytearray(keyframe.read_bytes())
data[-20] ^= 1
edited.write_bytes(data)

for path in (keyframe, edited):
    key = spec_hash(model="video-model", prompt="slow push-in", seed=0, params={}, inputs=[path])
    print(path.name, key[:16])

What the cache doesn't cover:

- a request that is still running,
- a crash in the middle of a generation,
- two processes asking for the same spec at once.

## Return references, not bytes

The tool returns an `AssetRef`: a URI, the spec hash, the size, the workload and the generation time.

- The PNG goes to the run folder, and to your bucket when `BUCKET` is set in `code/.env`.
- The model gets a line of text. The bytes never enter the conversation, so they never cost tokens on every later turn.

PydanticAI's own [image generation](https://pydantic.dev/articles/image-generation) does the opposite: the image comes back as `BinaryImage` and stays in the history. That's handy for one image and costly for many.

Only a step that has to look at the image, such as a judge, loads the bytes, once, in its own call.

In [ ]:
from tracing import last_tool_return

print("Model sees:", last_tool_return(result.all_messages()))
print("Run folder holds:", keyframe.stat().st_size, "bytes of PNG")

## Structured outputs for LLM steps

The plan step turns the brief into two prompts. Free text would need parsing. A typed output can't come back malformed.

`Plan` in `models.py`:

```python
class Plan(BaseModel):
    """The planner's output: one prompt per generation step."""

    image_prompt: str = Field(description="Prompt for one keyframe image that shows the shot")
    motion_prompt: str = Field(description="What moves in the 5-second clip made from it")
```

`output_type=Plan` makes the agent return a `Plan`, validated by Pydantic.
The same idea works for any LLM step whose answer code has to read, such as a judge's verdict.

In [ ]:
from models import Plan

model = (
    scripted_model(
        {
            "image_prompt": f"{brief.shot}, photorealistic, warm light",
            "motion_prompt": "slow push-in as the lamp switches on",
        }
    )
    if MODE == "fake"
    else token_factory_model()
)
planner = Agent(
    model,
    output_type=Plan,
    instructions="Turn the ad brief into a keyframe prompt and a motion prompt.",
)
plan = (await planner.run(brief.model_dump_json())).output
plan

## Notes

- A budget check doesn't belong inside the tool. Check the GPU budget before each paid generation, in the loop that calls the tool.
- Transport retries in `SanaEndpoint.generate()` cover only failures where the request never reached the model. Which errors the Sana template returns while loading: [verify: T4].
- Every demo writes its own run folder under `code/runs/02-multi-agent-media-workflows/`.